# T21 — Composite Presets

Generate multi-domain datasets using built-in presets with pre-configured shared entity mappings.

**What you'll learn:**
- List and inspect available presets
- Generate from a preset in one line
- Verify cross-domain FK integrity
- Compare preset vs ad-hoc composite

## 1. List Available Presets

In [1]:
from sqllocks_spindle.presets import list_presets, get_preset

for p in list_presets():
    print(f"{p.name:25s} domains={', '.join(p.domains)}")
    print(f"{'':25s} {p.description}\n")

enterprise                domains=retail, hr, financial
                          Enterprise dataset combining retail, HR, and financial domains

healthcare_system         domains=healthcare, insurance, hr
                          Healthcare system with insurance and HR

smart_factory             domains=manufacturing, iot, supply_chain
                          Smart factory combining manufacturing, IoT, and supply chain

digital_commerce          domains=retail, marketing, financial
                          Digital commerce with retail, marketing, and financial data

campus                    domains=education, hr
                          University campus combining education and HR

telecom_bundle            domains=telecom, marketing, financial
                          Telecom provider with marketing and billing



## 2. Generate from Enterprise Preset

In [2]:
from sqllocks_spindle import Spindle, RetailDomain, HrDomain, FinancialDomain
from sqllocks_spindle.domains.composite import CompositeDomain

preset = get_preset("enterprise")
composite = CompositeDomain(
    domains=[RetailDomain(), HrDomain(), FinancialDomain()],
    shared_entities=preset.shared_entities,
)

result = Spindle().generate(domain=composite, scale="small", seed=42)
print(result.summary())
print(f"\nTables: {sorted(result.table_names)}")

Spindle Generation Result
Schema: composite_financial_hr_retail
Domain: composite
Mode:   3nf
Seed:   42
Time:   0.7s

Table                             Rows  Columns
---------------------------------------------
financial_branch                   200        9
financial_customer               1,000       10
financial_loan                     400        9
financial_loan_payment           4,800        7
financial_transaction_category           40        4
hr_department                       30        4
hr_position                         80        6
hr_employee                        500       11
financial_account                2,200        7
financial_card                   1,760        8
financial_statement             13,200        8
financial_transaction           10,000        9
financial_fraud_flag               200        6
hr_compensation                  1,500        6
hr_performance_review            1,250        7
hr_termination                      75        6
hr_time_off_re

## 3. Verify Cross-Domain FK Integrity

In [3]:
errors = result.verify_integrity()
print(f"FK integrity errors: {len(errors)}")
for e in errors[:5]:
    print(f"  {e}")

# Check tables by domain
for prefix in ['retail_', 'hr_', 'financial_']:
    tables = [t for t in result.table_names if t.startswith(prefix)]
    total_rows = sum(len(result[t]) for t in tables)
    print(f"\n{prefix[:-1]:12s}: {len(tables)} tables, {total_rows:,} rows")

FK integrity errors: 2
  retail_customer.customer_id has 500 orphan FK values not found in hr_employee.employee_id
  financial_account.account_id has 1700 orphan FK values not found in hr_employee.employee_id

retail      : 9 tables, 21,850 rows

hr          : 9 tables, 8,035 rows

financial   : 10 tables, 33,800 rows


## 4. Inspect Shared Entities

In [4]:
print("Enterprise preset shared entities:")
for concept, config in preset.shared_entities.items():
    print(f"\n  {concept}:")
    print(f"    Primary: {config['primary']}")
    if 'links' in config:
        for domain, link in config['links'].items():
            print(f"    {domain}: {link}")

Enterprise preset shared entities:

  person:
    Primary: hr.employee
    retail: customer.customer_id
    financial: account.account_id


## 5. Try Other Presets

In [5]:
from sqllocks_spindle.domains.healthcare import HealthcareDomain
from sqllocks_spindle.domains.insurance import InsuranceDomain

hc_preset = get_preset("healthcare_system")
hc_composite = CompositeDomain(
    domains=[HealthcareDomain(), InsuranceDomain(), HrDomain()],
    shared_entities=hc_preset.shared_entities,
)
hc_result = Spindle().generate(domain=hc_composite, scale="small", seed=42)
print(f"Healthcare System: {len(hc_result.table_names)} tables, "
      f"{sum(len(hc_result[t]) for t in hc_result.table_names):,} rows")

Healthcare System: 27 tables, 50,627 rows


## 6. CLI Equivalent

```bash
# List presets
spindle presets

# Generate from preset
spindle composite enterprise --scale small --output ./data/ --format parquet

# Ad-hoc combination
spindle composite retail+hr+financial --scale small --output ./data/
```